# News classification model training
A simple text classification model, one of the classic BERT implementations. That's why in the next lines we're gonna import a magic pretrained model, that we'll only need to fine-tune.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

Let's read the dataset and define the model's hyperparameters

In [ ]:
id2labels = {0: 'Бизнес', 1: 'Наука', 2: 'Общество и происшествия', 3: 'Политика', 4: 'Спорт', 5: 'Технологии'}
labels2id = {k: v for v,k in id2labels.items()}


NCModel = BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels=6)
tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

data = pd.read_csv('news.csv')
train_texts, val_texts, train_labels, val_labels = train_test_split([h.strip()+' '+t.strip()[:300] for t,h in zip(data['text'], data['headline'])], [labels2id[c] for c in data['category']], random_state=42, test_size=0.2)



loss_fn = nn.CrossEntropyLoss()
learning_rate = 1e-4
optimizer = torch.optim.Adam(params=NCModel.parameters(), lr=learning_rate)
num_epochs = 10

Training loop.

In [ ]:

class NewsDataset(Dataset):
    def __init__(self, x, y):
        self.x = [tokenizer(t, padding="max_length", truncation=True, max_length=128, return_tensors='pt') for t in x]
        self.y = y
        del x,y
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
train_ds = NewsDataset(train_texts, train_labels)
val_ds = NewsDataset(val_texts, val_labels)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=8)
val_dl = DataLoader(val_ds, batch_size = 16)

for epoch in range(1, num_epochs+1):
    count = 0
    train_loss = 0
    for i, data in enumerate(train_dl):
        print(i+1)

        x,y = data
        output = NCModel(x['input_ids'].squeeze(1), x['attention_mask']).logits
        
        optimizer.zero_grad()

        loss = loss_fn(output, y)
        train_loss += loss.item()
        count += 1
        loss.backward()

        optimizer.step()

    print(f"\nTraining loss after epoch №{epoch}: {train_loss/count}")
    
    count = 0
    val_loss = 0
    for x,y in val_dl:
        with torch.no_grad:
            output = NCModel(**x).logits
        
        val_loss += loss_fn(output, y)
        count += 1
    print(f"Validation loss after epoch №{epoch}: {val_loss/count}")
        
        


In [ ]:
torch.save(NCModel, 'NCModel.pt')